In [4]:
#### without box

In [18]:
import os
import glob
import time
import gc
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import timedelta
from PIL import Image
from ultralytics import YOLO
import torch
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from shapely.geometry import Polygon
from shapely.ops import unary_union
from sklearn.cluster import DBSCAN
from pyproj import Transformer
import cv2
import pandas as pd
import numpy as np
import json
from shapely.geometry import Polygon
from sklearn.cluster import DBSCAN

# ======================== Drawing ========================
def draw_annotations_on_image(full_image_array, slices, object_infos, output_path):
    """
    Overlays bounding boxes, masks, and lat/lon centers with:
    - Transparent masks (alpha=0.3)
    - Class/conf labels
    - Red dot + lat/lon text
    - JPEG output
    """
    import cv2
    import numpy as np
    import json
    from rasterio.transform import rowcol

    # Convert rasterio (C,H,W) to OpenCV (H,W,C)
    img = np.moveaxis(full_image_array[:3], 0, -1)
    img = np.clip((img / img.max()) * 255, 0, 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    overlay = img.copy()

    # Offset lookup
    slice_offsets = {tuple(slice_data["offset_xy"]): slice_data for slice_data in slices}

    for obj in object_infos:
        tile_name = obj['tile_name']
        offset_xy = (int(obj['global_x'] - obj['x_center']), int(obj['global_y'] - obj['y_center']))

        # Bounding box
        x1_global = int(obj['bbox_x1'] + offset_xy[0])
        y1_global = int(obj['bbox_y1'] + offset_xy[1])
        x2_global = int(obj['bbox_x2'] + offset_xy[0])
        y2_global = int(obj['bbox_y2'] + offset_xy[1])

        conf = obj['confidence']
        cls = obj['class']
        label = f"Cls:{cls} ({conf:.2f})"

        # Draw bounding box
        cv2.rectangle(overlay, (x1_global, y1_global), (x2_global, y2_global), (0, 255, 0), 3)

        # Label text
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 2.5
        thickness = 5
        text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
        text_origin = (x1_global, max(y1_global - 10, text_size[1]))
        cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness + 2)
        cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)

        # Mask overlay
        try:
            mask_points = np.array(json.loads(obj['mask_points']), dtype=np.int32)
            mask_points_global = mask_points + np.array([offset_xy[0], offset_xy[1]])

            if len(mask_points_global) >= 3:
                mask_img = np.zeros_like(img)
                cv2.fillPoly(mask_img, [mask_points_global], (0, 0, 255))
                cv2.polylines(mask_img, [mask_points_global], isClosed=True, color=(0, 0, 255), thickness=2)
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.7, 0)
        except:
            pass

        # Lat/Lon center overlay
        try:
            lat = float(obj['latitude'])
            lon = float(obj['longitude'])

            # Use reverse transform to get row,col (y,x)
            for slice_data in slices:
                transform = slice_data['transform']
                try:
                    row, col = rowcol(transform, lon, lat)
                    pixel_x = col + slice_data["offset_xy"][0]
                    pixel_y = row + slice_data["offset_xy"][1]
                    break
                except:
                    continue

            # Draw center point
            cv2.circle(overlay, (int(pixel_x), int(pixel_y)), 8, (0, 0, 255), -1)

            # Add lat/lon text
            ll_label = f"({lat:.5f}, {lon:.5f})"
            cv2.putText(overlay, (int(pixel_x + 10), int(pixel_y - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 3)
            cv2.putText(overlay, (int(pixel_x + 10), int(pixel_y - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        except:
            continue

    # Save as JPEG
    output_path_jpg = output_path.replace(".png", ".jpg")
    cv2.imwrite(output_path_jpg, overlay)
    print(f"📸 Annotated image saved to: {output_path_jpg}")

# ======================== Sliding ========================
from rasterio.windows import Window
from rasterio.transform import from_origin

def slide_geotiff(geotiff_path, window_size, stride):
    """
    Splits a GeoTIFF into window_size×window_size tiles with given stride,
    **and** adds extra overlapping tiles on the right/bottom so that no pixel is left out.
    """
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width

        # 1) Compute all x-starts
        if width <= window_size:
            x_starts = [0]
        else:
            x_starts = list(range(0, width - window_size + 1, stride))
            # leftover on right?
            if x_starts[-1] + window_size < width:
                x_starts.append(width - window_size)

        # 2) Compute all y-starts
        if height <= window_size:
            y_starts = [0]
        else:
            y_starts = list(range(0, height - window_size + 1, stride))
            # leftover on bottom?
            if y_starts[-1] + window_size < height:
                y_starts.append(height - window_size)

        # 3) Build tiles for every (x,y) pair
        for y in y_starts:
            for x in x_starts:
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)
                new_transform = from_origin(
                    src.transform.c + x * src.transform.a,
                    src.transform.f + y * src.transform.e,
                    src.transform.a,
                    -src.transform.e
                )
                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": src.crs,
                    "position": (y, x),
                    "offset_xy": (x, y)
                })

    return tiles


# ======================== Per-Tile Suppression ========================
from torchvision.ops import nms
def maximum_likelihood_suppression(results, conf_threshold=0.7, iou_threshold=0.5):
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results
    boxes = results.boxes.xyxy
    scores = results.boxes.conf
    keep = scores >= conf_threshold
    if keep.sum() == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results
    keep_indices = nms(boxes[keep], scores[keep], iou_threshold)
    indices_to_keep = torch.nonzero(keep).squeeze(1)[keep_indices]
    results.boxes.data = results.boxes.data[indices_to_keep]
    if results.masks and results.masks.data is not None:
        results.masks.data = results.masks.data[indices_to_keep]
    return results

# ======================== Per-Tile Extraction ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image, offset_xy):
    object_data = []
    masks_out = []

    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return object_data, masks_out

    for idx, (mask, box, cls, conf) in enumerate(zip(
        results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf)):

        cls = int(cls.item())
        conf = float(conf.item())
        x1, y1, x2, y2 = map(float, box.tolist())

        mask_pixel_coords = np.array(mask.tolist(), dtype=np.float32)
        mask_gis_coords = []
        for px, py in mask_pixel_coords:
            try:
                lon, lat = rasterio.transform.xy(transform, py, px)
                mask_gis_coords.append([lon, lat])
            except:
                continue

        pixel_count = len(mask_pixel_coords)
        area = pixel_count * (transform.a * abs(transform.e))
        local_cx = (x1 + x2) / 2
        local_cy = (y1 + y2) / 2
        global_x = offset_xy[0] + local_cx
        global_y = offset_xy[1] + local_cy
        lon_center, lat_center = rasterio.transform.xy(transform, local_cy, local_cx)
        lon_x1, lat_y1 = rasterio.transform.xy(transform, y1, x1)
        lon_x2, lat_y2 = rasterio.transform.xy(transform, y2, x2)
        bbox_width_geo = abs(lon_x2 - lon_x1)
        bbox_height_geo = abs(lat_y2 - lat_y1)

        object_data.append({
            "latitude": lat_center,
            "longitude": lon_center,
            "transform": transform,
            "bbox_lat_y1": lat_y1,
            "bbox_lon_x1": lon_x1,
            "bbox_lat_y2": lat_y2,
            "bbox_lon_x2": lon_x2,
            "bbox_width_geo": bbox_width_geo,
            "bbox_height_geo": bbox_height_geo,
            "global_x": global_x,
            "global_y": global_y,
            "x_center": local_cx,
            "y_center": local_cy,
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
            "bbox_width": x2 - x1,
            "bbox_height": y2 - y1,
            "area_geo": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": conf,
            "tile_name": tif_image,
            "mask_index": idx,
            "mask_points": json.dumps(mask_pixel_coords.tolist()),
            "mask_points_gis": json.dumps(mask_gis_coords)
        })

        masks_out.append(mask_pixel_coords)

    return object_data, masks_out

#---------------------------------------------centroid calculation
def detect_poultry_farms(object_infos, eps_house=50, min_samples=2, eps_merge=100):
    # Filter out entries missing coordinates
    filtered_infos = [
        o for o in object_infos
        if 'latitude' in o and 'longitude' in o and pd.notnull(o['latitude']) and pd.notnull(o['longitude'])
    ]
    if not filtered_infos:
        return pd.DataFrame(columns=['merged_farm_id', 'longitude', 'latitude', 'num_houses'])
    df = pd.DataFrame(filtered_infos)
    coords = df[['longitude', 'latitude']].values
    dbscan = DBSCAN(eps=eps_house, min_samples=min_samples).fit(coords)
    df['initial_cluster'] = dbscan.labels_
    clustered = df[df['initial_cluster'] != -1].copy()
    if len(clustered) == 0:
        return pd.DataFrame(columns=['merged_farm_id', 'longitude', 'latitude', 'num_houses'])
    centroids = clustered.groupby('initial_cluster')[['longitude', 'latitude']].mean().reset_index()
    centroid_coords = centroids[['longitude', 'latitude']].values
    merge_cluster = DBSCAN(eps=eps_merge, min_samples=1).fit(centroid_coords)
    centroids['merged_farm_id'] = merge_cluster.labels_
    cluster_map = dict(zip(centroids['initial_cluster'], centroids['merged_farm_id']))
    clustered['merged_farm_id'] = clustered['initial_cluster'].map(cluster_map)
    farms = clustered.groupby('merged_farm_id')[['longitude', 'latitude']].mean().reset_index()
    farms['num_houses'] = clustered.groupby('merged_farm_id').size().values
    return farms


def lonlat_to_pixel(transform, lon, lat):
    # Returns (row, col) for rasterio window indexing
    from rasterio.transform import rowcol
    row, col = rowcol(transform, lon, lat)
    return row, col

def center_crop_window(row, col, img_width, img_height, window_size):
    """Returns a rasterio.Window such that the window is centered on (col, row)."""
    half = window_size // 2
    left = max(col - half, 0)
    top = max(row - half, 0)
    width = min(window_size, img_width - left)
    height = min(window_size, img_height - top)
    return Window(left, top, width, height), (left, top)

def run_inference_centroid_targeted(
    folder_path, model_path, final_csv_path, window_size=3000, batch_size=16,
    eps_house=50, min_samples=2, eps_merge=100
):
    model = YOLO(model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    all_cropped_object_infos = []

    for tif_image in tqdm(tif_images, desc="GeoTIFFs"):
        print(f"Processing {tif_image}")
        # 1. Initial sliding window inference for farm clustering
        initial_object_infos = []
        with rasterio.open(tif_image) as src:
            img_width, img_height = src.width, src.height
            slices = []
            stride = window_size  # full tiling, non-overlapping
            for y in range(0, img_height, stride):
                for x in range(0, img_width, stride):
                    w = min(window_size, img_width - x)
                    h = min(window_size, img_height - y)
                    window = Window(x, y, w, h)
                    img = src.read(window=window)
                    # Ensure 3 channels for YOLO
                    if img.shape[0] > 3:
                        img = img[:3]
                    elif img.shape[0] < 3:
                        img = np.tile(img, (3 // img.shape[0], 1, 1))
                    pil_img = Image.fromarray(np.moveaxis(img, 0, -1))
                    with torch.inference_mode():
                        results = model([pil_img], device=device, batch=1, verbose=False)[0]
                    results = maximum_likelihood_suppression(results)
                    offset_xy = (x, y)
                    object_info, _ = overlay_segmentation_on_array(
                        results, src.window_transform(window), None, os.path.basename(tif_image), offset_xy)
                    initial_object_infos.extend(object_info)

            # 2. Farm clustering
            farm_centroids_df = detect_poultry_farms(initial_object_infos, eps_house, min_samples, eps_merge)
            if farm_centroids_df.empty:
                print("No farms detected in this image.")
                continue

            # 3. For each centroid, crop, run inference, map results back
            for idx, farm in farm_centroids_df.iterrows():
                lon, lat = farm['longitude'], farm['latitude']
                try:
                    row, col = rasterio.transform.rowcol(src.transform, lon, lat)
                except Exception as e:
                    print(f"Could not convert lon/lat to pixel: {e}")
                    continue
                window, (left, top) = center_crop_window(row, col, img_width, img_height, window_size)
                crop_img = src.read(window=window)
                if crop_img.shape[0] > 3:
                    crop_img = crop_img[:3]
                elif crop_img.shape[0] < 3:
                    crop_img = np.tile(crop_img, (3 // crop_img.shape[0], 1, 1))
                pil_crop = Image.fromarray(np.moveaxis(crop_img, 0, -1))
                with torch.inference_mode():
                    results_crop = model([pil_crop], device=device, batch=1, verbose=False)[0]
                results_crop = maximum_likelihood_suppression(results_crop)
                offset_xy = (left, top)
                crop_object_info, _ = overlay_segmentation_on_array(
                    results_crop, src.window_transform(window), None,
                    f"{os.path.basename(tif_image)}_farm{idx}", offset_xy)
                all_cropped_object_infos.extend(crop_object_info)

    pd.DataFrame(all_cropped_object_infos).to_csv(final_csv_path, index=False)
    print(f"✅ Targeted centroid-based results saved to {final_csv_path}")


# ======================== Full Pipeline Runner ========================
def run_inference_centroid_targeted(
    folder_path, model_path, final_csv_path, window_size=3000, batch_size=16,
    eps_house=50, min_samples=2, eps_merge=100
):
    model = YOLO(model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    print(f"Using device: {device}")

    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    all_cropped_object_infos = []

    for tif_image in tqdm(tif_images, desc="GeoTIFFs"):
        # 1. Run initial sliding window inference to get all object_infos for farm detection
        slices = slide_geotiff(tif_image, window_size, window_size)  # stride=window_size for full tiles
        initial_object_infos = []
        for batch in [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]:
            pil_imgs_batch = [Image.fromarray(np.moveaxis(s["img_array"][:3], 0, -1)) for s in batch]
            with torch.inference_mode():
                results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)
            for slice_data, results in zip(batch, results_batch):
                results = maximum_likelihood_suppression(results)
                object_info, _ = overlay_segmentation_on_array(
                    results, slice_data["transform"], None, os.path.basename(tif_image), slice_data["offset_xy"])
                initial_object_infos.extend(object_info)

        # 2. Cluster into farms using centroids
        farm_centroids_df = detect_poultry_farms(initial_object_infos, eps_house, min_samples, eps_merge)
        if farm_centroids_df.empty:
            continue

        # 3. For each farm centroid, crop a window, run inference, map back to image coordinates
        with rasterio.open(tif_image) as src:
            for idx, farm in farm_centroids_df.iterrows():
                lon, lat = farm['longitude'], farm['latitude']
                # Convert lon/lat to pixel
                try:
                    row, col = rasterio.transform.rowcol(src.transform, lon, lat)
                except Exception as e:
                    print(f"⚠️ Could not locate farm centroid in image: {e}")
                    continue

                # Center window on (row, col)
                half_w = window_size // 2
                win_col_off = max(col - half_w, 0)
                win_row_off = max(row - half_w, 0)
                win_width = min(window_size, src.width - win_col_off)
                win_height = min(window_size, src.height - win_row_off)
                window = Window(win_col_off, win_row_off, win_width, win_height)
                crop_img = src.read(window=window)
                crop_transform = src.window_transform(window)

                # Run inference on the cropped image
                pil_crop = Image.fromarray(np.moveaxis(crop_img[:3], 0, -1))
                with torch.inference_mode():
                    results_crop = model([pil_crop], device=device, batch=1, verbose=False)[0]
                o = maximum_likelihood_suppression(results_crop)

                # Use the farm index/id as the tile_name for tracking (optional)
                offset_xy = (win_col_off, win_row_off)
                crop_object_info, _ = overlay_segmentation_on_array(
                    results_crop, crop_transform, None,
                    f"{os.path.basename(tif_image)}", offset_xy)
                all_cropped_object_infos.extend(crop_object_info)

    # Save results
    pd.DataFrame(all_cropped_object_infos).to_csv(final_csv_path, index=False)
    print(f"✅ Targeted centroid-based results saved to {final_csv_path}")



In [22]:
# ================= SAM Setup =================
from segment_anything import sam_model_registry, SamPredictor

# SAM configuration
CHECKPOINT_PATH = "../weights/sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)
sam_predictor = SamPredictor(sam)

# ================= YOLO + SAM Integrated Pipeline =================
# ================= YOLO + SAM Integrated Pipeline =================
# ================= YOLO + SAM Integrated Pipeline =================
def run_inference_centroid_targeted(
    folder_path, model_path, final_csv_path, window_size=3000, batch_size=16,
    eps_house=50, min_samples=2, eps_merge=100
):
    model = YOLO(model_path).to(DEVICE)

    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    all_cropped_object_infos = []

    for tif_image in tqdm(tif_images, desc="GeoTIFFs"):
        slices = slide_geotiff(tif_image, window_size, window_size)
        initial_object_infos = []

        for batch in [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]:
            pil_imgs_batch = [Image.fromarray(np.moveaxis(s["img_array"][:3], 0, -1)) for s in batch]
            results_batch = model(pil_imgs_batch, device=DEVICE, batch=batch_size, verbose=False)
            for slice_data, results in zip(batch, results_batch):
                # results = maximum_likelihood_suppression(results)
                object_info, _ = overlay_segmentation_on_array(
                    results, slice_data["transform"], None, os.path.basename(tif_image), slice_data["offset_xy"])
                initial_object_infos.extend(object_info)

        # farm_centroids_df = detect_poultry_farms(initial_object_infos, eps_house, min_samples, eps_merge)
        # if farm_centroids_df.empty:
            # continue

        # with rasterio.open(tif_image) as src:
        #     for idx, farm in farm_centroids_df.iterrows():
        #         lon, lat = farm['longitude'], farm['latitude']
        #         row, col = rasterio.transform.rowcol(src.transform, lon, lat)

        #         half_w = window_size // 2
        #         win_col_off = max(col - half_w, 0)
        #         win_row_off = max(row - half_w, 0)
        #         win_width = min(window_size, src.width - win_col_off)
        #         win_height = min(window_size, src.height - win_row_off)
        #         window = Window(win_col_off, win_row_off, win_width, win_height)
        #         crop_img = src.read(window=window)
        #         crop_transform = src.window_transform(window)

        #         # YOLO Inference
        #         pil_crop = Image.fromarray(np.moveaxis(crop_img[:3], 0, -1))
        #         results_crop = model([pil_crop], device=DEVICE, batch=1, verbose=False)[0]
        #         results_crop = maximum_likelihood_suppression(results_crop)

                # # SAM Inference (POINT PROMPT ONLY, NO BOX PROMPT)
                # sam_predictor.set_image(np.array(pil_crop))
                # sam_object_infos = []

                # for box in results_crop.boxes.xyxy.cpu().numpy():
                #     x1, y1, x2, y2 = box
                #     # Foreground: box center, Background: top-left corner
                #     cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                #     input_point = np.array([
                #         [cx, cy],   # Foreground
                #         [x1, y1]    # Background
                #     ])
                #     input_label = np.array([1, 0])  # 1=foreground, 0=background

                #     masks, scores, _ = sam_predictor.predict(
                #         point_coords=input_point,
                #         point_labels=input_label,
                #         multimask_output=True
                #         # box=... <--- REMOVED!
                #     )
                #     best_mask = masks[np.argmax(scores)]

                #     # Convert mask to polygon
                #     contours, _ = cv2.findContours(best_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                #     if contours:
                #         largest_contour = max(contours, key=cv2.contourArea)
                #         polygon = largest_contour.reshape(-1, 2)

                #         mask_points_global = polygon + np.array([win_col_off, win_row_off])
                #         mask_gis_coords = [
                #             rasterio.transform.xy(src.transform, pt[1], pt[0]) for pt in mask_points_global
                #         ]

                #         sam_object_infos.append({
                #             "mask_points": json.dumps(polygon.tolist()),
                #             "mask_points_gis": json.dumps(mask_gis_coords)
                #         })

                # # Combine YOLO & SAM outputs
                # offset_xy = (win_col_off, win_row_off)
                # crop_object_info, _ = overlay_segmentation_on_array(
                #     results_crop, crop_transform, None,
                #     f"{os.path.basename(tif_image)}", offset_xy)

                # for yolo_obj, sam_obj in zip(crop_object_info, sam_object_infos):
                #     yolo_obj.update(sam_obj)
                # all_cropped_object_infos.extend(crop_object_info)

    pd.DataFrame(initial_object_infos).to_csv(final_csv_path, index=False)
    # pd.DataFrame(all_cropped_object_infos).to_csv(final_csv_path, index=False)
    print(f"✅ Targeted centroid-based results saved to {final_csv_path}")





# Run Example
run_inference_centroid_targeted(
    folder_path="../test_sample/",
    model_path="C:/Users/mza0288/Documents/CODE/runs/segment/yolo11l-500-sgd-04092025/weights/best.pt",
    final_csv_path="CSV_Yolo_test_SAM_final/final_only_yolo_without_mls.csv",
    window_size=3000, batch_size=32
)


C:\Users\mza0288\AppData\Local\anaconda3\envs\torchenv\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.

✅ Targeted centroid-based results saved to CSV_Yolo_test_SAM_final/final_only_yolo_without_mls.csv


In [24]:
# ================= YOLO-ONLY Pipeline (SAM REMOVED) =================
def run_inference_centroid_targeted(
    folder_path, model_path, final_csv_path, window_size=3000, batch_size=16,
    eps_house=50, min_samples=2, eps_merge=100
):
    model = YOLO(model_path).to(DEVICE)

    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    all_cropped_object_infos = []

    for tif_image in tqdm(tif_images, desc="GeoTIFFs"):
        slices = slide_geotiff(tif_image, window_size, window_size)
        initial_object_infos = []

        for batch in [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]:
            pil_imgs_batch = [Image.fromarray(np.moveaxis(s["img_array"][:3], 0, -1)) for s in batch]
            results_batch = model(pil_imgs_batch, device=DEVICE, batch=batch_size, verbose=False)

            for slice_data, results in zip(batch, results_batch):
                results = maximum_likelihood_suppression(results)
                object_info, _ = overlay_segmentation_on_array(
                    results,
                    slice_data["transform"],
                    None,
                    os.path.basename(tif_image),
                    slice_data["offset_xy"]
                )
                initial_object_infos.extend(object_info)

        farm_centroids_df = detect_poultry_farms(
            initial_object_infos, eps_house, min_samples, eps_merge
        )
        if farm_centroids_df.empty:
            continue

        with rasterio.open(tif_image) as src:
            for _, farm in farm_centroids_df.iterrows():
                lon, lat = farm["longitude"], farm["latitude"]
                row, col = rasterio.transform.rowcol(src.transform, lon, lat)

                half_w = window_size // 2
                win_col_off = max(col - half_w, 0)
                win_row_off = max(row - half_w, 0)
                win_width = min(window_size, src.width - win_col_off)
                win_height = min(window_size, src.height - win_row_off)

                window = Window(win_col_off, win_row_off, win_width, win_height)
                crop_img = src.read(window=window)
                crop_transform = src.window_transform(window)

                # YOLO Inference
                pil_crop = Image.fromarray(np.moveaxis(crop_img[:3], 0, -1))
                results_crop = model([pil_crop], device=DEVICE, batch=1, verbose=False)[0]
                # results_crop = maximum_likelihood_suppression(results_crop)

                # YOLO-only overlay and object info extraction
                offset_xy = (win_col_off, win_row_off)
                crop_object_info, _ = overlay_segmentation_on_array(
                    results_crop,
                    crop_transform,
                    None,
                    f"{os.path.basename(tif_image)}",
                    offset_xy
                )

                all_cropped_object_infos.extend(crop_object_info)

    pd.DataFrame(all_cropped_object_infos).to_csv(final_csv_path, index=False)
    print(f"✅ Targeted centroid-based results saved to {final_csv_path}")


# Run Example
run_inference_centroid_targeted(
    folder_path="../test_sample/",
    model_path="C:/Users/mza0288/Documents/CODE/runs/segment/yolo11l-500-sgd-04092025/weights/best.pt",
    final_csv_path="CSV_Yolo_test_SAM_final/final_only_yolo_without_mls_1.csv",
    window_size=3000,
    batch_size=32
)


GeoTIFFs: 100%|████████████████████████████████████████████████████████████████████████| 10/10 [15:32<00:00, 93.25s/it]


✅ Targeted centroid-based results saved to CSV_Yolo_test_SAM_final/final_only_yolo_without_mls_1.csv


In [28]:
import os
import cv2
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm
import ast

def lonlat_to_pixel(lon, lat, transform):
    row, col = rasterio.transform.rowcol(transform, lon, lat)
    return int(col), int(row)  # (x, y) in pixel space

def parse_mask_points(mask_str, transform):
    try:
        coords = ast.literal_eval(mask_str)
        pixels = [lonlat_to_pixel(lon, lat, transform) for lon, lat in coords]
        return np.array(pixels, dtype=np.int32)
    except Exception as ex:
        print(f"[Error] Mask parse failed: {ex} | {mask_str}")
        return None

def visualize_geo_annotations(
    csv_path,
    geotiff_dir,
    output_dir,
    tile_col="tile_name",
    point_lon_col="longitude",
    point_lat_col="latitude",
    box_y1_col="bbox_lat_y1",
    box_x1_col="bbox_lon_x1",
    box_y2_col="bbox_lat_y2",
    box_x2_col="bbox_lon_x2",
    mask_col="mask_points_gis"
):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    df[tile_col] = df[tile_col].astype(str)

    tiff_files = {os.path.basename(f): os.path.join(geotiff_dir, f)
                  for f in os.listdir(geotiff_dir) if f.lower().endswith(('.tif', '.tiff'))}

    for tiff_file, full_path in tqdm(tiff_files.items()):
        if tiff_file not in df[tile_col].values:
            print(f"[Skip] {tiff_file} not found in {tile_col} column")
            continue
        objects = df[df[tile_col] == tiff_file].to_dict(orient='records')
        if not objects:
            print(f"[Skip] No objects for {tiff_file}")
            continue

        with rasterio.open(full_path) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0))
            transform = src.transform

        if img.dtype != np.uint8:
            img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
        overlay = img.copy()

        for idx, obj in enumerate(objects):
            try:
                # Point
                lon = float(obj[point_lon_col])
                lat = float(obj[point_lat_col])
                px, py = lonlat_to_pixel(lon, lat, transform)
                cv2.circle(overlay, (px, py), 6, (0, 0, 255), -1)  # Red point

                # Box
                box_x1 = float(obj[box_x1_col])
                box_y1 = float(obj[box_y1_col])
                box_x2 = float(obj[box_x2_col])
                box_y2 = float(obj[box_y2_col])
                x1, y1 = lonlat_to_pixel(box_x1, box_y1, transform)
                x2, y2 = lonlat_to_pixel(box_x2, box_y2, transform)
                x1, x2 = np.clip([x1, x2], 0, overlay.shape[1] - 1)
                y1, y2 = np.clip([y1, y2], 0, overlay.shape[0] - 1)
                x1, x2 = sorted([int(x1), int(x2)])
                y1, y2 = sorted([int(y1), int(y2)])
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)  # Green box

                # # Mask
                # mask_coords = parse_mask_points(obj[mask_col], transform)
                # if mask_coords is not None and len(mask_coords) > 2:
                #     cv2.polylines(overlay, [mask_coords], isClosed=True, color=(255, 0, 0), thickness=2)  # Blue mask

                # print(f"[{tiff_file}][{idx}] Box: ({x1},{y1})-({x2},{y2}), Point: ({px},{py}), Mask N={len(mask_coords) if mask_coords is not None else 0}")
            except Exception as ex:
                print(f"[Error] {tiff_file}, object {idx}: {ex}")
                continue

        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        out_path = os.path.join(output_dir, os.path.splitext(tiff_file)[0] + "_geo_debug.jpg")
        cv2.imwrite(out_path, overlay_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    print(f"[Saved] {out_path}")

# --- USAGE ---
visualize_geo_annotations(
    csv_path="CSV_Yolo_test_SAM_final/final.csv",
    geotiff_dir="../test_sample",
    output_dir="CSV_Yolo_test_SAM_final/CSV_Yolo_test_first/rect"
)


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [06:36<00:00, 39.62s/it]


[Saved] CSV_Yolo_test_SAM_final/CSV_Yolo_test_first/rect\m_3108504_sw_16_030_20211114_geo_debug.jpg


In [5]:
##### without box

In [6]:
import os
import cv2
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm
import torch
from segment_anything import sam_model_registry, SamPredictor

# =========== SAM SETUP ===========
CHECKPOINT_PATH = "weights/sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)
sam_predictor = SamPredictor(sam)

def lonlat_to_pixel(lon, lat, transform):
    row, col = rasterio.transform.rowcol(transform, lon, lat)
    return int(col), int(row)  # (x, y) in pixel space

def run_sam_geo_csv_batch(
    csv_path,
    geotiff_dir,
    output_dir,
    tile_col="tile_name",
    point_lon_col="longitude",
    point_lat_col="latitude",
    box_y1_col="bbox_lat_y1",
    box_x1_col="bbox_lon_x1",
    box_y2_col="bbox_lat_y2",
    box_x2_col="bbox_lon_x2",
    class_label_col="class",
    conf_col="confidence"
):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    df[tile_col] = df[tile_col].astype(str)
    tiff_files = {os.path.basename(f): os.path.join(geotiff_dir, f)
                  for f in os.listdir(geotiff_dir) if f.lower().endswith(('.tif', '.tiff'))}

    for tiff_file, full_path in tqdm(tiff_files.items()):
        if tiff_file not in df[tile_col].values:
            print(f"[Skip] {tiff_file} not found in {tile_col} column")
            continue

        objects = df[df[tile_col] == tiff_file].to_dict(orient='records')
        if not objects:
            print(f"[Skip] No objects for {tiff_file}")
            continue

        with rasterio.open(full_path) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0))
            transform = src.transform

        if img.dtype != np.uint8:
            img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
        overlay = img.copy()

        for idx, obj in enumerate(objects):
            try:
                # ----- Foreground Point Prompt -----
                lon = float(obj[point_lon_col])
                lat = float(obj[point_lat_col])
                px, py = lonlat_to_pixel(lon, lat, transform)

                # ----- Box Prompt -----
                box_x1 = float(obj[box_x1_col])
                box_y1 = float(obj[box_y1_col])
                box_x2 = float(obj[box_x2_col])
                box_y2 = float(obj[box_y2_col])
                x1, y1 = lonlat_to_pixel(box_x1, box_y1, transform)
                x2, y2 = lonlat_to_pixel(box_x2, box_y2, transform)
                x1, x2 = np.clip([x1, x2], 0, overlay.shape[1] - 1)
                y1, y2 = np.clip([y1, y2], 0, overlay.shape[0] - 1)
                x1, x2 = sorted([int(x1), int(x2)])
                y1, y2 = sorted([int(y1), int(y2)])

                # ----- Crop region -----
                if x2 <= x1 or y2 <= y1:
                    print(f"[Skip] Invalid box for {tiff_file}, object {idx}")
                    continue
                roi_rgb = overlay[y1:y2, x1:x2]
                if roi_rgb.shape[0] < 2 or roi_rgb.shape[1] < 2:
                    print(f"[Skip] Crop too small for {tiff_file}, object {idx}")
                    continue

                # Foreground prompt: point inside object (relative to crop)
                rel_px = px - x1
                rel_py = py - y1

                # Background prompt: top-left corner of the box (relative to crop)
                rel_x1 = 0  # x1 - x1
                rel_y1 = 0  # y1 - y1

                # Combine both prompts for SAM: [[px, py], [x1, y1]]
                input_point = np.array([
                    [rel_px, rel_py],   # foreground
                    [rel_x1, rel_y1]    # background (box corner)
                ])
                input_label = np.array([1, 0])  # 1=foreground, 0=background

                rel_box = np.array([0, 0, x2 - x1, y2 - y1])

                sam_predictor.set_image(roi_rgb)
                masks, scores, logits = sam_predictor.predict(
                    point_coords=input_point,
                    point_labels=input_label,
                    # box=rel_box,
                    multimask_output=True
                )
                if masks is None or len(masks) == 0:
                    print(f"[Skip] No mask from SAM for {tiff_file}, object {idx}")
                    continue
                mask = masks[np.argmax(scores)]

                # Place mask in full image
                mask_full = np.zeros(overlay.shape[:2], dtype=np.uint8)
                mask_resized = cv2.resize(mask.astype(np.uint8), (x2 - x1, y2 - y1), interpolation=cv2.INTER_NEAREST)
                mask_full[y1:y2, x1:x2] = mask_resized

                # Draw
                mask_img = np.zeros_like(overlay)
                mask_img[mask_full > 0] = [0, 0, 255]
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.4, 0)
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
                # Foreground point
                cv2.circle(overlay, (int(px), int(py)), 4, (0, 255, 255), -1)
                # Background point (box corner)
                cv2.circle(overlay, (x1, y1), 4, (255, 128, 0), -1)
                label = f"Cls:{obj.get(class_label_col, '')} ({float(obj.get(conf_col, 0.0)):.2f})"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 1.0
                thickness = 2
                text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
                text_origin = (x1, max(y1 - 5, text_size[1]))
                cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness + 2)
                cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)

            except Exception as ex:
                print(f"[Error] {tiff_file}, object {idx}: {ex}")
                continue

        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        out_path = os.path.join(output_dir, os.path.splitext(tiff_file)[0] + "_sam_annotated.jpg")
        cv2.imwrite(out_path, overlay_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
        print(f"[Saved] {out_path}")

# ========== USAGE ==========
run_sam_geo_csv_batch(
    csv_path="CSV_Yolo_test_SAM_4/final.csv",
    geotiff_dir="test_sample",
    output_dir="CSV_Yolo_test_SAM_4/test"
)


 10%|████████                                                                         | 1/10 [06:42<1:00:26, 402.92s/it]

[Saved] CSV_Yolo_test_SAM_4/test/m_3108502_sw_16_030_20211103_sam_annotated.jpg


 20%|████████████████▏                                                                | 2/10 [27:54<2:01:53, 914.19s/it]

[Saved] CSV_Yolo_test_SAM_4/test/m_3108501_se_16_030_20211103_sam_annotated.jpg


 30%|████████████████████████▎                                                        | 3/10 [34:12<1:18:04, 669.22s/it]

...

[Saved] CSV_Yolo_test_SAM_4/test/m_3108502_ne_16_030_20211103_sam_annotated.jpg


 90%|████████████████████████████████████████████████████████████████████████▉        | 9/10 [1:00:59<06:56, 416.52s/it]

[Saved] CSV_Yolo_test_SAM_4/test/m_3108501_ne_16_030_20211103_sam_annotated.jpg


100%|████████████████████████████████████████████████████████████████████████████████| 10/10 [1:02:21<00:00, 374.18s/it]

[Saved] CSV_Yolo_test_SAM_4/test/m_3108501_nw_16_030_20211103_sam_annotated.jpg
